# Mathematical Engineering - Financial Engineering, FY 2025-2026

# Risk Management - Assignment 1: Hedging a Swaption Portfolio

**Case study:** The IR-derivative desk of Polimi Bank has the following positions opened today (15/02/2008):

- Long swaption payer 1m&10y 5y ATM - Notional €650 Mln
- Vanilla 10y IRS fixed rate receiver - Notional €550 Mln


In [1]:
# Importing the libraries
import numpy as np
import pandas as pd

import datetime as dt

from utilities.date_functions import (
    business_date_offset,
    date_series,
)
from utilities.ex0_utilities import bootstrap
from utilities.ex1_utilities import (
    swaption_price_calculator,
    irs_proxy_duration,
    swap_par_rate,
    swap_mtm,
    SwapType,
)

In [2]:
# =============================================================================
# Portfolio Parameters
# =============================================================================

# Swaption
swaption_maturity_y = 10  # Years component of swaption expiry
swaption_maturity_m = 0  # Months component of swaption expiry
swaption_tenor_y = 5  # Underlying swap tenor in years
swaption_fixed_leg_freq = 1  # Annual fixed leg payments
swaption_type = SwapType.PAYER
swaption_notional = 650_000_000
sigma_black = 0.7955  # Black implied volatility for the swaption

# IRS
irs_maturity = 10  # In years
irs_fixed_leg_freq = 1  # Annual fixed leg payments
irs_notional = 550_000_000
irs_type = SwapType.PAYER

In [3]:
# =============================================================================
# Bootstrap the discount curve from market data (Assignment 0)
# =============================================================================

# !!! COMPLETE AS APPROPRIATE !!!

today = dt.date(2025,2,15)

dt = pd.read_csv('mkt_data/dt.csv',
                index_col = 'Market',
                usecols = ['Market','TARGET'],
                converters = {'TARGET':pd.to_datetime})
settlement_date  = dt.TARGET['Settlement']

depo_converter = lambda x: float(x)/100.0  # apply the correct one

df_depos = pd.read_csv('mkt_data/depos.csv', 
                   index_col ='Depos',
                   usecols = ['Depos','ASK','BID'], 
                   converters={'Depos':pd.to_datetime,'BID':depo_converter,'ASK':depo_converter})

future_converter = lambda x: float(x) # apply the correct one

futures = pd.read_csv('mkt_data/futures.csv',
                      index_col ='Futures',
                      usecols = ['Futures','ASK','BID'],
                      converters={'Futures':pd.to_datetime,'BID':future_converter,'ASK':future_converter})
expiry = pd.read_csv('mkt_data/expiry.csv',
                     index_col = 'Futures',
                     usecols =['Futures', 'Settle', 'Expiry'], 
                     converters = {'Futures':pd.to_datetime, 'Settle':pd.to_datetime, 'Expiry':pd.to_datetime})
df_futures = futures.join(expiry)

swap_converter = lambda x: float(x) # apply the correct one

df_swaps = pd.read_csv('mkt_data/swaps.csv',
                    index_col = 'Swaps',
                    usecols = ['Swaps','BID','ASK'],
                    converters={'Swaps':pd.to_datetime,'BID':swap_converter,'ASK':swap_converter})


settlement_date = dt.TARGET['Settlement']

discount_factors, zero_rates = bootstrap(settlement_date, df_depos, df_futures, df_swaps)

## Q1: Mark-to-Market the portfolio at the mid-rate curve


In [4]:
# =============================================================================
# Q1: Compute the forward swap rate (= swaption strike, since ATM)
# =============================================================================

# Swaption expiry: today + 10y1m
swaption_expiry = business_date_offset(
    today, year_offset=swaption_maturity_y, month_offset=swaption_maturity_m
)

# Underlying swap expiry: swaption expiry + 5y tenor
underlying_expiry = business_date_offset(
    today,
    year_offset=swaption_maturity_y + swaption_tenor_y,
    month_offset=swaption_maturity_m,
)

# Fixed leg schedule of the underlying forward-starting swap
swaption_underlying_fixed_leg_schedule = date_series(
    swaption_expiry, underlying_expiry, swaption_fixed_leg_freq
)

# Forward swap rate
fwd_swap_rate = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors,
    swaption_underlying_fixed_leg_schedule[0],
)
print(f"Forward swap rate: {fwd_swap_rate:.3%}")

Forward swap rate: 1.560%


In [ ]:
# =============================================================================
# Q1: Portfolio MtM
# =============================================================================

strike = fwd_swap_rate # !!! COMPLETE AS APPROPRIATE !!!

swaption_price, swaption_delta = swaption_price_calculator(
    fwd_swap_rate,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors,
    swaption_type,
    compute_delta=True,
)

irs_expiry = business_date_offset(today, year_offset=irs_maturity)
irs_fixed_leg_payment_dates = date_series(today, irs_expiry, irs_fixed_leg_freq)[1:]

irs_rate = swap_par_rate(irs_fixed_leg_payment_dates, discount_factors)
irs_mtm = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors
)

# Portfolio MtM = swaption value + IRS value
ptf_mtm = swaption_notional * swaption_price + irs_notional * irs_mtm
print(f"Swaption price (per unit notional): {swaption_price:.6f}")
print(f"IRS MtM (per unit notional):        {irs_mtm:.6f}")
print(f"Portfolio MtM:                       \u20ac{ptf_mtm:,.2f}")

TypeError: bad operand type for unary -: 'NoneType'

## Q2: Evaluate the portfolio DV01-parallel


In [ ]:
# =============================================================================
# Q2: Portfolio DV01-parallel (numerical)
# =============================================================================

discount_factors_up = None # !!! COMPLETE AS APPROPRIATE !!!

fwd_swap_rate_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under the shocked curve
swaption_price_up = swaption_price_calculator(
    fwd_swap_rate,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_up,
    swaption_type,
)

irs_mtm_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors, swap_type=irs_type
)

# Shocked portfolio MtM
ptf_mtm_up = swaption_notional * swaption_price_up + irs_notional * irs_mtm_up

# DV01-parallel
ptf_numeric_dv01 = ptf_mtm_up - ptf_mtm
print(f"Portfolio DV01-parallel: \u20ac{ptf_numeric_dv01:,.2f}")

## Q3: Analytical portfolio DV01 approximation


In [ ]:
# =============================================================================
# Q3: Analytical portfolio DV01
# =============================================================================

irs_duration = irs_proxy_duration(
    today, irs_rate, irs_fixed_leg_payment_dates, discount_factors
)

ptf_proxy_dv01 = (
    swaption_notional * swaption_delta + irs_notional * irs_duration
) * 1e-4

print(f"Portfolio proxy DV01:    \u20ac{ptf_proxy_dv01:,.2f}")
print(f"Portfolio numeric DV01:  \u20ac{ptf_numeric_dv01:,.2f}")
print(f"Difference:             \u20ac{ptf_proxy_dv01 - ptf_numeric_dv01:,.2f}")

## Q4: Delta-hedge the portfolio with a 10y IRS


In [ ]:
# =============================================================================
# Q4: Delta hedging with a 10y IRS
# =============================================================================

min_lot = 1_000_000  # IRS traded in multiples of €1M

delta_hedge_swap_notional = None # !!! COMPLETE AS APPROPRIATE !!!

# Verify: hedged portfolio DV01 should be ~0
delta_hedge_dv01 = (
    swaption_notional * swaption_price_up + delta_hedge_swap_notional * irs_mtm_up
) - ptf_mtm
print(
    f"Numerical hedge: \u20ac{delta_hedge_swap_notional:,.0f} total IRS notional, residual DV01: \u20ac{delta_hedge_dv01:,.0f}"
)

# --- Analytical hedge (using the proxy DV01) ---
delta_hedge_swap_notional_proxy = None # !!! COMPLETE AS APPROPRIATE !!!
print(
    f"Analytical hedge: \u20ac{delta_hedge_swap_notional_proxy:,.0f} total IRS notional"
)
print(
    "\nThe analytical approximation significantly overestimates the required hedge notional."
)

## Q5: Portfolio coarse-grained bucket DV01 (10y and 15y buckets)


In [ ]:
# =============================================================================
# Q5: Coarse-Grained Bucket DV01 construction
# =============================================================================

# !!! COMPLETE AS APPROPRIATE !!!
discount_factors_10y_up = None
discount_factors_15y_up = None

In [ ]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 10y Bucket DV01
# =============================================================================

swaption_price_10y_up = swaption_price_calculator(
    fwd_swap_rate,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_10y_up,
    swaption_type,
)

# IRS MtM under 10y bucket shock
irs_mtm_10y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_10y_up, swap_type=irs_type
)

ptf_mtm_10y_up = (
    swaption_notional * swaption_price_10y_up + irs_notional * irs_mtm_10y_up
)

ptf_numeric_10y_dv01 = ptf_mtm_10y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 10y Bucket DV01: \u20ac{ptf_numeric_10y_dv01:,.2f}")

In [ ]:
# =============================================================================
# Q5: Portfolio Coarse-Grained 15y Bucket DV01
# =============================================================================

fwd_swap_rate_15y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_15y_up,
    swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price under 15y bucket shock
swaption_price_15y_up = swaption_price_calculator(
    fwd_swap_rate_15y_up,
    strike,
    today,
    swaption_expiry,
    underlying_expiry,
    sigma_black,
    swaption_fixed_leg_freq,
    discount_factors_15y_up,
    swaption_type,
)

# IRS MtM under 15y bucket shock
irs_mtm_15y_up = swap_mtm(
    irs_rate, irs_fixed_leg_payment_dates, discount_factors_15y_up,
)

ptf_mtm_15y_up = (
    swaption_notional * swaption_price_15y_up
    + irs_notional * irs_mtm_15y_up
)

ptf_numeric_15y_dv01 = ptf_mtm_15y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 15y Bucket DV01: \u20ac{ptf_numeric_15y_dv01:,.2f}")

## Q6: Delta-hedge with two liquid IRS (10y and 15y)


In [ ]:
# =============================================================================
# Q6: Delta hedging with two IRS (10y and 15y)
# =============================================================================
# !!! COMPLETE AS APPROPRIATE !!!

## Q7: Curve flattening scenario


In [ ]:
# =============================================================================
# Q7: Curve flattening scenario
# =============================================================================
# !!! COMPLETE AS APPROPRIATE !!!